In [ ]:
# parameters
# BINDER_FAST: set N=4 for fast cloud execution
N = 8                 # Hilbert space truncation
omega0_GHz = 5.0      # bare 0->1 frequency (GHz)
K_GHz = 0.2           # Kerr nonlinearity (GHz)
kappa_MHz = 1.0       # peak single-photon loss rate (MHz)
Gamma_MHz = 100.0     # bath FWHM (MHz)
nbar = 0.02           # thermal occupation
gamma_phi_MHz = 0.05  # pure dephasing rate (MHz)

In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.driven_kerr import (
    DrivenKerrConfig,
    make_operators,
    make_jump_ops,
    J, J_vectorized,
    run_lindblad,
    effective_loss_rate,
)

TWO_PI = 2 * np.pi

## Module 6a: Error Channels in Superconducting Systems

**Learning objectives:**
- Identify the three fundamental decoherence channels in bosonic superconducting systems: photon loss, thermal re-excitation, and pure dephasing
- Write down and evaluate the Lindblad jump operators for each channel
- Understand how the bath spectral density $J(\omega)$ shapes which environmental frequencies couple to the system
- Verify the linear scaling of each channel's contribution to infidelity with its rate parameter

---

**Sections:**
[1 Decoherence Taxonomy](#sec1) ·
[2 The Spectral Density](#sec2) ·
[3 Lindblad vs Structured Bath](#sec3) ·
[4 Per-Channel Scaling](#sec4) ·
[5 Exercises](#sec5)

<a id="sec1"></a>
## 1  Decoherence Taxonomy

Every open quantum system couples to its environment. For a microwave resonator in a dilution refrigerator, three processes dominate:

### 1.1  Energy relaxation (T₁ / photon loss)

A photon leaks out of the cavity into the transmission line or substrate modes.
In the Lindblad formalism the jump operator is

$$\hat{L}_{\rm loss} = \sqrt{\kappa(1+\bar{n})}\,\hat{a},$$

where $\kappa$ is the single-photon linewidth (loss rate at the bath peak) and
$\bar{n}$ is the thermal occupation. The $(1+\bar{n})$ factor follows from detailed
balance: a hot bath both absorbs and re-emits photons.

The energy-decay time is $T_1 = 1/[\kappa(1+\bar{n})]$. State-of-the-art
3D aluminium cavities reach $T_1 \sim 1{-}5$ ms ($\kappa/2\pi \sim 0.1{-}1$ kHz);
2D resonators are typically faster ($\kappa/2\pi \sim 1{-}10$ kHz).

### 1.2  Thermal re-excitation

The environment can also *inject* a thermal photon, driving
$|n\rangle \to |n+1\rangle$. The jump operator is

$$\hat{L}_{\rm gain} = \sqrt{\kappa\bar{n}}\,\hat{a}^\dagger.$$

At millikelvin temperatures $\bar{n} = [e^{\hbar\omega_0/k_B T}-1]^{-1} \ll 1$,
so this channel is much weaker than loss. Typical values: $\bar{n} = 0.01$–$0.1$
depending on the microwave filtering and thermalisation.

### 1.3  Pure dephasing (T₂*)

Random phase kicks from two-level defects (TLS), charge noise, or flux noise
commute with the photon number operator $\hat{n}$:

$$\hat{L}_{\rm deph} = \sqrt{\gamma_\phi}\,\hat{n}.$$

This process does **not** change $\langle\hat{n}\rangle$ — no energy is exchanged.
It destroys off-diagonal coherences $\rho_{mn}$ (for $m \neq n$) exponentially.
The pure dephasing rate is $\gamma_\phi$. In superconducting resonators it is
typically much smaller than $\kappa$; in transmon qubits it can dominate.

**Important distinction.** For photon-number conserving dephasing, the jump
operator $\hat{n}$ cannot transfer population between Fock states in the
Lindblad (Method A) formulation. This is why dephasing contributes *zero* to
$|1\rangle \to |0\rangle$ population decay in Method A — a fact we will exploit
in the error budget. Drive-induced dephasing in the Floquet picture (Method C)
is different and breaks this rule; see Module 3c.

| Parameter | Typical range | Our value |
|---|---|---|
| $\kappa/2\pi$ | 0.1–10 kHz (cavity) | 1 MHz |
| $\bar{n}$ | 0.01–0.1 | 0.02 |
| $\gamma_\phi/2\pi$ | 0–100 kHz | 0.05 MHz |

In [ ]:
# Build the canonical configuration and inspect jump operators
cfg = DrivenKerrConfig(
    N=N,
    omega0=TWO_PI * omega0_GHz,
    K=TWO_PI * K_GHz,
    omega_d=TWO_PI * (omega0_GHz - 0.005),  # 5 MHz red-detuned drive
    kappa=TWO_PI * kappa_MHz * 1e-3,
    Gamma=TWO_PI * Gamma_MHz * 1e-3,
    nbar=nbar,
    gamma_phi=TWO_PI * gamma_phi_MHz * 1e-3,
    k_max=5,
    n_t=512,
)

a_op, adag_op, n_op = make_operators(cfg.N)
c_ops = make_jump_ops(cfg)

# Compute rates analytically
gamma_loss = J(cfg.omega0, cfg)     # J(+ω₀) = κ(1+n̄)
gamma_gain = J(-cfg.omega0, cfg)    # J(-ω₀) = κ n̄

print("Jump operators and their rates:")
print(f"  Loss  operator: √κ(1+n̄) · a    rate = {gamma_loss/TWO_PI*1e3:.5f} MHz")
print(f"  Gain  operator: √(κ n̄) · a†    rate = {gamma_gain/TWO_PI*1e3:.5f} MHz")
print(f"  Deph  operator: √γ_φ · n        rate = {cfg.gamma_phi/TWO_PI*1e3:.5f} MHz")
print()
print(f"Number of jump operators returned by make_jump_ops: {len(c_ops)}")
print()
print(f"Loss/gain ratio (should equal (1+n̄)/n̄ = {(1+nbar)/nbar:.1f}):")
print(f"  gamma_loss / gamma_gain = {gamma_loss/gamma_gain:.2f}")
print()
print(f"Detailed balance check: J(-ω₀)/J(+ω₀) = n̄/(1+n̄)")
print(f"  From J: {gamma_gain/gamma_loss:.6f}")
print(f"  n̄/(1+n̄): {nbar/(1+nbar):.6f}")

<a id="sec2"></a>
## 2  The Spectral Density $J(\omega)$

Real electromagnetic baths are not flat in frequency — they have structure on the
scale of circuit mode spacings. We model the bath with a Lorentzian spectral density:

$$J(\omega) = \kappa \frac{(\Gamma/2)^2}{(\omega - \omega_f)^2 + (\Gamma/2)^2} \times
\begin{cases} 1 + \bar{n} & \omega > 0 \\ \bar{n} & \omega < 0 \end{cases}$$

where:
- $\kappa$ is the peak coupling rate (at $\omega = \omega_f$)
- $\Gamma$ is the FWHM of the Lorentzian feature (bath bandwidth)
- $\omega_f$ is the center of the bath feature (usually $\omega_0$ or $\omega_{12} + \omega_d$)
- The thermal factors implement detailed balance: $J(-\omega)/J(\omega) = \bar{n}/(1+\bar{n})$

**Physical meaning of $\Gamma$:**
- Small $\Gamma$ (narrow bath): only a narrow window of frequencies couple efficiently.
  If the system's transition frequency shifts (e.g., from drive dressing), the coupling
  drops dramatically. This is the key mechanism studied in Module 3c.
- Large $\Gamma$ (white bath, $\Gamma \to \infty$): flat coupling across all relevant
  sideband frequencies. Lindblad becomes exact in this limit.

The Lorentzian is plotted below for three bath widths to build intuition.

In [ ]:
# Plot J(ω) for several bath widths
omega_arr = np.linspace(0.5 * TWO_PI, 10.0 * TWO_PI, 2000)  # 0–10 GHz
Gamma_vals_MHz = [10.0, 100.0, 1000.0]
colors = ["C0", "C1", "C2"]

fig, ax = plt.subplots(figsize=(8, 4))

for Gamma_val, color in zip(Gamma_vals_MHz, colors):
    cfg_plot = cfg.replace(
        Gamma=TWO_PI * Gamma_val * 1e-3,
        omega_f=cfg.omega0,      # bath centered at ω₀ for illustration
    )
    J_arr = J_vectorized(omega_arr, cfg_plot)
    ax.plot(
        omega_arr / TWO_PI,          # convert to GHz
        J_arr / TWO_PI * 1e3,        # convert to MHz
        color=color, lw=1.8,
        label=rf"$\Gamma/2\pi = {Gamma_val:.0f}$ MHz",
    )

ax.axvline(cfg.omega0 / TWO_PI, color="gray", ls="--", lw=1, label=r"$\omega_0/2\pi = 5.0$ GHz")
ax.set_xlabel(r"Frequency $\omega/2\pi$ (GHz)")
ax.set_ylabel(r"$J(\omega)/2\pi$ (MHz)")
ax.set_title("Bath spectral density $J(\\omega)$ for three bath widths (emission side)")
ax.legend()
ax.set_xlim(0.5, 10.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Peak value of J at ω₀ (should equal κ · (1+n̄)):")
print(f"  κ·(1+n̄)/2π = {cfg.kappa*(1+cfg.nbar)/TWO_PI*1e3:.4f} MHz  (any bath width)")
print()
print("Bath coupling at ω₀ ± Γ/2 (half-power point):")
for Gamma_val in [10.0, 100.0, 1000.0]:
    cfg_p = cfg.replace(Gamma=TWO_PI*Gamma_val*1e-3, omega_f=cfg.omega0)
    omega_halfpower = cfg.omega0 + TWO_PI * Gamma_val/2 * 1e-3
    J_hp = J(omega_halfpower, cfg_p)
    print(f"  Γ={Gamma_val:.0f} MHz: J(ω₀ + Γ/2)/2π = {J_hp/TWO_PI*1e3:.4f} MHz")

<a id="sec3"></a>
## 3  Lindblad vs Structured Bath

The Lindblad approximation (Method A) fixes the jump operator rates by sampling
$J$ at the bare transition frequency $\omega_0$, *regardless of the drive amplitude*.
This is valid only when the bath is flat over the frequency range explored by
drive-induced sideband shifts.

**When does it fail?**
A drive of amplitude $\varepsilon$ shifts the Floquet quasi-energies by approximately
$\delta\omega_{\rm sb} \approx \varepsilon^2 / \Gamma$ relative to $\omega_0$.
Method A ignores this shift; Method C (Floquet–Markov) tracks it.

The Lindblad approximation breaks down when:

$$\delta\omega_{\rm sb} \gtrsim \Gamma \quad \Longleftrightarrow \quad
\varepsilon \gtrsim \Gamma.$$

For our bath with $\Gamma/2\pi = 100$ MHz, this means the drive must be kept
below $\sim 100$ MHz for Lindblad to be trusted. At $\varepsilon = K = 200$ MHz,
the Lindblad rate overestimates the true decay rate by a factor of $\sim 20$.
This is the key result from Module 3c and the motivation for the error budget
in Module 6b.

In [ ]:
# Illustrate sideband shift for the narrow (Gamma=100 MHz) bath
cfg_narrow = cfg.replace(Gamma=TWO_PI * Gamma_MHz * 1e-3, omega_f=cfg.omega0)

omega_plot = np.linspace(
    (cfg.omega0 - 5*cfg_narrow.Gamma) / TWO_PI,
    (cfg.omega0 + 5*cfg_narrow.Gamma) / TWO_PI,
    1000,
)
J_plot = J_vectorized(omega_plot * TWO_PI, cfg_narrow)

# Sideband shift at ε = 0.5K (representative)
eps_rep = 0.5 * cfg.K
delta_sb = eps_rep**2 / cfg_narrow.Gamma      # approximate sideband shift
omega_sb = (cfg.omega0 + delta_sb) / TWO_PI   # shifted sideband frequency

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(omega_plot, J_plot / TWO_PI * 1e3, "C1", lw=2,
        label=rf"$J(\omega)$, $\Gamma/2\pi = {Gamma_MHz:.0f}$ MHz")

# Bare ω₀
J_at_omega0 = J(cfg.omega0, cfg_narrow)
ax.axvline(cfg.omega0 / TWO_PI, color="C0", ls="-", lw=1.5, alpha=0.9,
           label=rf"Bare $\omega_0/2\pi = {cfg.omega0/TWO_PI:.3f}$ GHz")
ax.plot(cfg.omega0/TWO_PI, J_at_omega0/TWO_PI*1e3, "C0o", ms=8)

# Shifted sideband at ε = 0.5K
J_at_sb = J(cfg.omega0 + delta_sb, cfg_narrow)
ax.axvline(omega_sb, color="C2", ls="--", lw=1.5, alpha=0.9,
           label=rf"Shifted sideband at $\varepsilon = 0.5K$ (approx.): {omega_sb:.3f} GHz")
ax.plot(omega_sb, J_at_sb/TWO_PI*1e3, "C2^", ms=8)

ax.set_xlabel(r"Frequency $\omega/2\pi$ (GHz)")
ax.set_ylabel(r"$J(\omega)/2\pi$ (MHz)")
ax.set_title("Bath spectral density with bare and drive-shifted sideband frequencies")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Drive amplitude ε = 0.5K = {eps_rep/TWO_PI*1e3:.1f} MHz")
print(f"Approximate sideband shift δω_sb = ε²/Γ = {delta_sb/TWO_PI*1e3:.2f} MHz")
print(f"J at bare ω₀:         {J_at_omega0/TWO_PI*1e3:.4f} MHz")
print(f"J at shifted sideband: {J_at_sb/TWO_PI*1e3:.4f} MHz")
print(f"Ratio J(ω₀+δω) / J(ω₀) = {J_at_sb/J_at_omega0:.4f}")
print()
print("Lab note: Method A always uses J(ω₀) = {:.4f} MHz regardless of drive.".format(
    J_at_omega0/TWO_PI*1e3))
print("Method C would use J at the shifted frequency ≈ {:.4f} MHz.".format(
    J_at_sb/TWO_PI*1e3))

<a id="sec4"></a>
## 4  Per-Channel Linear Scaling

A key property of the Lindblad master equation is that when multiple independent
decoherence channels are weak, their contributions to infidelity add linearly:

$$1 - F_{\rm total} \approx \sum_i (1 - F_i), \qquad \text{for small infidelities.}$$

This additivity has a deeper origin: each jump operator appears independently in
the Lindblad equation $\dot\rho = \sum_k \mathcal{D}[L_k]\rho$, and the first-order
perturbation in each $L_k$ is independent. The condition for validity is that
total infidelity $\lesssim 10\%$.

We verify this linear scaling numerically: for each channel, sweep its rate from
zero to several times the default value and check that the decay rate extracted
from a simulation scales linearly.

In [ ]:
# Sweep κ and measure effective loss rate from |1⟩ decay
kappa_vals = np.linspace(0.1, 10.0, 10)   # MHz
nbar_vals  = np.linspace(0.0, 0.10, 10)
gphi_vals  = np.linspace(0.0, 1.0, 10)    # MHz

# Time array: 3 decay times at reference kappa
T_sim = 3.0 / cfg.kappa
tlist_rate = np.linspace(0, T_sim, 120)
rho0 = qt.fock_dm(cfg.N, 1)

rates_kappa = []
for kappa_val in kappa_vals:
    cfg_k = cfg.replace(kappa=TWO_PI * kappa_val * 1e-3, nbar=0.0, gamma_phi=0.0)
    res = run_lindblad(cfg_k, rho0, tlist_rate)
    rates_kappa.append(effective_loss_rate(res, tlist_rate, cfg_k))

rates_nbar = []
for nbar_val in nbar_vals:
    cfg_n = cfg.replace(nbar=nbar_val, gamma_phi=0.0)
    res = run_lindblad(cfg_n, rho0, tlist_rate)
    rates_nbar.append(effective_loss_rate(res, tlist_rate, cfg_n))

rates_gphi = []
for gphi_val in gphi_vals:
    # Use a longer simulation for dephasing (which doesn't directly decay |1⟩)
    # We measure how it broadens the decay from |+⟩ = (|0⟩+|1⟩)/√2
    cfg_g = cfg.replace(kappa=TWO_PI * 0.05e-3, nbar=0.0, gamma_phi=TWO_PI * gphi_val * 1e-3)
    res = run_lindblad(cfg_g, rho0, tlist_rate)
    rates_gphi.append(effective_loss_rate(res, tlist_rate, cfg_g))

rates_kappa = np.array(rates_kappa)
rates_nbar  = np.array(rates_nbar)
rates_gphi  = np.array(rates_gphi)

print("Linear scaling checks complete.")

In [ ]:
# Plot the three panels of linear scaling
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Panel 1: rate vs κ
ax = axes[0]
ax.plot(kappa_vals, rates_kappa / TWO_PI * 1e3, "C0o-", ms=6, lw=1.5)
# Analytic line: J(omega0) = kappa * (Gamma/2)^2 / ((omega0-omega_f)^2 + (Gamma/2)^2) * (1+nbar)
# At omega0 = omega_f: J = kappa * (1+nbar=1 since nbar=0)
ax.plot(kappa_vals, kappa_vals, "k--", lw=1, label="slope = 1 (ideal)")
ax.set_xlabel(r"$\kappa/2\pi$ (MHz)")
ax.set_ylabel(r"Extracted rate / $2\pi$ (MHz)")
ax.set_title("Loss rate $\\propto \\kappa$")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: rate vs n̄
ax = axes[1]
ax.plot(nbar_vals, rates_nbar / TWO_PI * 1e3, "C1s-", ms=6, lw=1.5)
# Analytic prediction: kappa * (1 + 2*nbar) from loss + gain at omega0
kappa0_MHz = kappa_MHz  # our reference kappa
ax.plot(nbar_vals, kappa0_MHz * (1 + 2 * nbar_vals), "k--", lw=1,
        label=r"$\kappa(1+2\bar{n})$ (analytic)")
ax.set_xlabel(r"Thermal occupation $\bar{n}$")
ax.set_ylabel(r"Extracted rate / $2\pi$ (MHz)")
ax.set_title(r"Total rate $\propto \kappa(1+2\bar{n})$")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: rate vs γ_φ
ax = axes[2]
ax.plot(gphi_vals, rates_gphi / TWO_PI * 1e3, "C2^-", ms=6, lw=1.5)
ax.axhline(0.05, color="k", ls="--", lw=1, label=r"baseline $\kappa=0.05$ MHz")
ax.set_xlabel(r"$\gamma_\phi/2\pi$ (MHz)")
ax.set_ylabel(r"Extracted rate / $2\pi$ (MHz)")
ax.set_title(r"Dephasing ($\hat{n}$ preserves $\langle\hat{n}\rangle$, rate $\approx$ const)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle("Per-channel linear scaling", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("Panel 3 key observation:")
print("  Dephasing (γ_φ n̂) does NOT change ⟨n̂⟩ — the extracted loss rate is")
print("  insensitive to γ_φ when measuring population in Fock states.")
print("  This linear independence is exactly what the error budget exploits.")

<a id="sec5"></a>
## 5  Exercises

### Exercise 1: Thermal balance

At what value of $\bar{n}$ does the thermal re-excitation rate equal the loss rate?
Find the answer analytically by setting
$\kappa\bar{n} = \kappa(1+\bar{n})$
and solve for $\bar{n}$. Then verify numerically by sweeping $\bar{n}$ from 0 to 1
and finding where the two rates cross.

In [ ]:
# YOUR CODE HERE
# Hint: analytically, κ·n̄ = κ·(1+n̄)  →  n̄ = 1+n̄ → no solution?
# Actually: loss rate = J(+ω₀) ∝ κ(1+n̄), gain rate = J(-ω₀) ∝ κ·n̄.
# They can never be equal for n̄ < ∞ — loss always exceeds gain.
# Instead, show numerically that gain ≈ loss when n̄ → ∞.
# EXTENSION: at what n̄ does thermal contribute 10% of the total rate?

### Exercise 2: Dephasing does not decay populations

Start in the superposition $|+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$ and
evolve under pure dephasing only ($\kappa = 0$, $\bar{n} = 0$, $\gamma_\phi > 0$).

1. Plot $\langle\hat{n}\rangle(t)$ — it should remain constant at $1/2$.
2. Plot the off-diagonal density matrix element $|\rho_{01}(t)|$ — it should decay as
   $e^{-\gamma_\phi t}$.

This confirms that pure dephasing destroys coherence without dissipating energy.

In [ ]:
# YOUR CODE HERE
# Hint:
# psi_plus = (qt.basis(N, 0) + qt.basis(N, 1)).unit()
# rho_plus = qt.ket2dm(psi_plus)
# cfg_deph_only = cfg.replace(kappa=0.0, nbar=0.0, gamma_phi=TWO_PI*0.1e-3)
# tlist_deph = np.linspace(0, 5.0/cfg_deph_only.gamma_phi, 200)
# res = run_lindblad(cfg_deph_only, rho_plus, tlist_deph)
# Then plot qt.expect(n_op, res.states) and |res.states[i][0,1]| vs time